## Calculate indices per split from the Vinyals Omniglot split

As used in the Prototypical networks paper.

In [1]:
# Download text from: https://github.com/jakesnell/prototypical-networks/blob/master/data/omniglot/splits/vinyals/test.txt

import urllib

test_url = "https://raw.githubusercontent.com/jakesnell/prototypical-networks/master/data/omniglot/splits/vinyals/test.txt"
val_url = "https://raw.githubusercontent.com/jakesnell/prototypical-networks/master/data/omniglot/splits/vinyals/val.txt"

with urllib.request.urlopen(test_url) as response:
    test_html = response.read()

with urllib.request.urlopen(val_url) as response:
    val_html = response.read()

# Parse text
test_entries = test_html.decode("utf-8").splitlines()
val_entries = val_html.decode("utf-8").splitlines()


In [2]:
# Only keep characters that end in rot000 and remove the rot000
test_entries = [x[:-6] for x in test_entries if x.endswith("rot000")]

In [3]:
# Only keep characters that end in rot000 and remove the rot000
val_entries = [x[:-6] for x in val_entries if x.endswith("rot000")]

In [4]:
import torchvision

In [5]:
background_set = torchvision.datasets.Omniglot('../datasets/data', background=True, download=True)
non_background_set = torchvision.datasets.Omniglot('../datasets/data', background=False, download=True)

Files already downloaded and verified
Files already downloaded and verified


In [6]:
from os.path import join

In [7]:
test_indices = []

for i in range(len(non_background_set)):
    image_name, character_class = non_background_set._flat_character_images[i]
    image_path = join(non_background_set.target_folder, non_background_set._characters[character_class], image_name)

    for entry in test_entries:
        if entry in image_path:
            test_indices.append(i + 964*20)
            break

In [8]:
val_indices = []

for i in range(len(background_set)):
    image_name, character_class = background_set._flat_character_images[i]
    image_path = join(background_set.target_folder, background_set._characters[character_class], image_name)

    for entry in val_entries:
        if entry in image_path:
            val_indices.append(i)
            break

In [9]:
# Train indices are all the remaining indices between 0 and 1623*20
train_indices = list(set(range(1623*20)) - set(val_indices) - set(test_indices))

## Save indices to file

In [ ]:
# Save the indices to a pickle file

import pickle

with open('../artifacts/omniglot_indices_vinyals_split.pkl', 'wb') as f:
    pickle.dump((train_indices, val_indices, test_indices), f)


## Example usage

In [12]:
import torch
from torch.utils.data import Subset

all_splits_data = torch.utils.data.ConcatDataset([background_set, non_background_set])

train_set = Subset(all_splits_data, train_indices)
val_set = Subset(all_splits_data, val_indices)
test_set = Subset(all_splits_data, test_indices)